In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

from khiops.sklearn import KhiopsClassifier, KhiopsEncoder
from pathlib import Path

from sklearn.utils.multiclass import type_of_target

In [ ]:
# Paramètres injectables par Papermill
DATA_PATH = "Adult.txt"
TARGET_COL = "class"
OUTPUT_DIR = "khiops_output_adult"
SEP = "\t"
ID_COL = None
N_SPLITS = 10
RANDOM_STATE = 42
OUTPUT_SEP = ";"

In [ ]:
DATASET_NAME = Path(DATA_PATH).stem
OUTPUT_DIR = Path(OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. LECTURE ET PREPARATION DES DONNEES
# ============================================================

df = pd.read_csv(DATA_PATH, sep=SEP)

# Nettoyage des noms de colonnes
df.columns = df.columns.str.strip()

# Vérification de la présence de la cible
if TARGET_COL not in df.columns:
    raise ValueError(
        f"La colonne cible '{TARGET_COL}' est absente. "
        f"Colonnes disponibles : {df.columns.tolist()}"
    )

# Nettoyage des colonnes texte :
# - suppression des espaces ;
# - "?" et valeurs vides considérées comme manquantes.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace({"?": pd.NA, "": pd.NA})
    )

# Suppression facultative des lignes sans cible :
# StratifiedKFold et les métriques ne peuvent pas travailler
# avec une cible manquante.
df = df.dropna(subset=[TARGET_COL]).copy()

# Gestion éventuelle de la colonne identifiant
if ID_COL is not None:
    if ID_COL not in df.columns:
        raise ValueError(
            f"La colonne identifiant '{ID_COL}' est absente. "
            f"Colonnes disponibles : {df.columns.tolist()}"
        )

    ids = df[ID_COL].copy()

    # L'identifiant ne doit pas être utilisé comme variable explicative
    df_model = df.drop(columns=[ID_COL]).copy()

else:
    # En l'absence d'identifiant métier, on conserve l'index original
    # afin d'identifier chaque ligne dans les fichiers exportés.
    ids = pd.Series(df.index, index=df.index, name="row_id")
    df_model = df.copy()

############
# Séparation des variables explicatives et de la cible

X = df_model.drop(columns=[TARGET_COL]).copy()

# ------------------------------------------------------------
# NORMALISATION DE LA CIBLE
#
# - Les cibles déjà numériques restent numériques :
#   Pima, Waveform, Pendigits, Shuttle, etc.
#
# - Les cibles catégorielles restent des chaînes :
#   Adult, Mushroom, Vehicle, Segmentation, Letters, Phoneme...
#
# Cette distinction est nécessaire car Khiops retourne les
# prédictions numériques sous forme d'entiers pour Pima.
# ------------------------------------------------------------

y_raw = df_model[TARGET_COL].copy()

if pd.api.types.is_numeric_dtype(y_raw):

    y_numeric = pd.to_numeric(y_raw, errors="raise")

    # Si les labels sont entiers, on conserve un int64.
    if (y_numeric.dropna() % 1 == 0).all():
        y = y_numeric.astype("int64")
    else:
        y = y_numeric.astype("float64")

else:

    y = pd.Series(
        y_raw
        .astype("string")
        .str.strip()
        .to_numpy(dtype=str),
        index=y_raw.index,
        name=TARGET_COL,
        dtype=object,
    )

y.name = TARGET_COL


# Conversion des colonnes texte vers le type Python object :
# cela évite les problèmes possibles entre Pandas string[python],
# scikit-learn et Khiops.
for col in X.columns:
    if (
        pd.api.types.is_string_dtype(X[col])
        or isinstance(X[col].dtype, pd.StringDtype)
    ):
        X[col] = X[col].astype(object)

print("\nContrôle de la cible pour scikit-learn :")
print(f" - dtype y : {y.dtype}")
print(f" - types Python détectés : {y.map(type).unique().tolist()}")
print(f" - classes : {sorted(y.unique().tolist())}")
print(f" - type détecté : {type_of_target(y)}")


# Vérifications
if X.empty:
    raise ValueError("Aucune variable explicative n'est disponible dans X.")

if y.nunique(dropna=True) < 2:
    raise ValueError(
        "La cible doit contenir au moins deux classes distinctes."
    )

print("Dimensions des données :")
print(f" - Nombre de lignes : {len(X):,}")
print(f" - Nombre de variables explicatives : {X.shape[1]}")

print("\nTypes des variables explicatives :")
print(X.dtypes.value_counts())

print("\nModalités de la cible :")
print(y.value_counts(dropna=False))

print("\nContrôle des objets créés :")
print(f" - X   : {X.shape}")
print(f" - y   : {y.shape}")
print(f" - ids : {ids.shape}")


In [ ]:
# ============================================================
# 3. VALIDATION CROISEE STRATIFIEE A 10 FOLDS
# ============================================================
#
# Pour chaque fold :
# - le classifieur Khiops est appris sur les 9 folds train ;
# - le recodeur Conditional Info est appris sur les 9 folds train ;
# - le test est transformé uniquement avec le recodeur du train ;
# - les poids SNB sont spécifiques au fold ;
# - tous les exports comportent une colonne "fold".
#

if y.value_counts().min() < N_SPLITS:
    raise ValueError(
        f"La classe la moins représentée contient seulement "
        f"{y.value_counts().min()} lignes. "
        f"Une validation croisée stratifiée à {N_SPLITS} folds "
        f"requiert au moins {N_SPLITS} observations par classe."
    )

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:
# ============================================================
# 4. FONCTIONS UTILITAIRES
# ============================================================

def normalize_name(name):
    """Normalise un nom pour faciliter la comparaison."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def infer_source_variable(encoded_column, original_columns):
    """
    Associe une colonne recodée Khiops à une variable source.

    Exemple :
        Info1Peducation_num -> education_num
    """
    encoded_norm = normalize_name(encoded_column)

    candidates = []

    for source_col in original_columns:
        source_norm = normalize_name(source_col)

        if source_norm and source_norm in encoded_norm:
            candidates.append(source_col)

    # Exemple : education_num doit être préféré à education.
    if candidates:
        return max(candidates, key=lambda x: len(normalize_name(x)))

    return None


def logsumexp(values, axis=1):
    """
    Calcul numériquement stable de :

        log(sum(exp(values)))

    nécessaire pour calculer log p_w(D) sans sous-dépassement
    numérique lorsque les scores sont très négatifs.
    """
    values = np.asarray(values, dtype=float)

    max_values = np.max(
        values,
        axis=axis,
        keepdims=True,
    )

    return np.squeeze(
        max_values
        + np.log(
            np.sum(
                np.exp(values - max_values),
                axis=axis,
                keepdims=True,
            )
        ),
        axis=axis,
    )


def infer_weighted_ci_columns_by_class_index(
    weighted_ci_columns,
    n_classes,
):
    """
    Regroupe les colonnes Conditional Info PONDEREES selon
    l'indice de classe Khiops.

    Exemples :
        weighted_Info1PSepalLength
        weighted_Info1PSepalWidth
        ...
        -> classe d'indice 1

        weighted_Info2PSepalLength
        ...
        -> classe d'indice 2

    Retour :
        {
            1: [... colonnes de la classe 1 ...],
            2: [... colonnes de la classe 2 ...],
            ...
        }
    """
    columns_by_class_index = {}

    pattern = re.compile(
        r"^weighted_Info(?P<class_index>\d+)P"
    )

    for column in weighted_ci_columns:
        match = pattern.match(column)

        if match is None:
            raise ValueError(
                "Format inattendu pour une colonne Conditional Info "
                "pondérée :\n"
                f" - {column}\n\n"
                "Format attendu, par exemple :\n"
                " - weighted_Info1PSepalLength"
            )

        class_index = int(match.group("class_index"))

        columns_by_class_index.setdefault(
            class_index,
            [],
        ).append(column)

    expected_indices = set(range(1, n_classes + 1))
    found_indices = set(columns_by_class_index.keys())

    if found_indices != expected_indices:
        raise ValueError(
            "Indices de classes incohérents dans les colonnes "
            "Conditional Info pondérées.\n\n"
            f"Indices attendus : {sorted(expected_indices)}\n"
            f"Indices trouvés  : {sorted(found_indices)}\n\n"
            "Colonnes disponibles :\n"
            + "\n".join(
                f" - {column}"
                for column in weighted_ci_columns
            )
        )

    return columns_by_class_index


def compute_weighted_global_log_probability(
    X_ci_weighted,
    class_priors,
    model_classes,
    columns_by_class_index,
):
    
    
    """
    Calcule log p_w(D) pour le SNB pondéré.

    Formule multiclasse :

        p_w(D)
        =
        somme_j p(C_j) * produit_m p(D_m | C_j) ** w_m

    soit, dans le domaine logarithmique :

        log p_w(D)
        =
        logsumexp_j [
            log p(C_j)
            +
            somme_m w_m log p(D_m | C_j)
        ]

    Important :
    ----------
    X_ci_weighted contient déjà les quantités :

        w_m * log p(D_m | C_j)

    Il ne faut donc PAS multiplier une seconde fois par les poids.
    """
    model_classes = list(model_classes)
    n_classes = len(model_classes)

    if len(class_priors) != n_classes:
        raise ValueError(
            "Nombre incohérent de probabilités a priori et de classes."
        )

    if class_priors.isna().any():
        raise ValueError(
            "Une ou plusieurs probabilités a priori sont manquantes."
        )

    if (class_priors <= 0).any():
        raise ValueError(
            "Une probabilité a priori nulle ou négative a été détectée."
        )

    log_priors = np.log(
        class_priors.to_numpy(dtype=float)
    )

    # Chaque colonne j correspond à :
    #
    # log p(C_j) + somme_m w_m log p(D_m | C_j)
    #
    log_joint_probabilities = np.zeros(
        (len(X_ci_weighted), n_classes),
        dtype=float,
    )

    for class_position, _class_name in enumerate(
        model_classes,
        start=1,
    ):
        class_columns = columns_by_class_index[class_position]
    
        # Les Conditional Info exportées sont positives et correspondent
        # à - w_m * log p(D_m | C_j).
        weighted_log_likelihood = -(
            X_ci_weighted[class_columns]
            .sum(axis=1)
            .to_numpy(dtype=float)
        )
    
        log_joint_probabilities[:, class_position - 1] = (
            log_priors[class_position - 1]
            + weighted_log_likelihood
        )

    # log p_w(D)
    weighted_global_log_probability = logsumexp(
        log_joint_probabilities,
        axis=1,
    )

    return pd.Series(
        weighted_global_log_probability,
        index=X_ci_weighted.index,
        name="global_log_p_D_weighted",
    )


In [ ]:
# ============================================================
# 5. LISTES DE STOCKAGE DES RESULTATS
# ============================================================

all_weights = []
all_mappings = []
all_metrics = []
all_train_recoded = []
all_test_recoded = []

# Nouveau : suivi des probabilités globales pondérées
# et des probabilités a priori de chaque fold.
all_global_probabilities = []
all_class_priors = []


In [ ]:
# ============================================================
# 6. BOUCLE SUR LES 10 FOLDS
# ============================================================

for fold_number, (train_index, test_index) in enumerate(
    cv.split(X, y),
    start=1,
):

    print("\n" + "=" * 75)
    print(f"FOLD {fold_number}/{N_SPLITS}")
    print("=" * 75)

    # --------------------------------------------------------
    # 6.1. Construction des jeux train / test du fold
    # --------------------------------------------------------

    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()

    y_train = y.iloc[train_index].copy()
    y_test = y.iloc[test_index].copy()

    id_train = ids.iloc[train_index].copy()
    id_test = ids.iloc[test_index].copy()

    print(f"Train fold {fold_number} : {X_train.shape[0]:,} lignes")
    print(f"Test  fold {fold_number} : {X_test.shape[0]:,} lignes")
    print(f"Variables initiales : {X_train.shape[1]}")

    fold_dir = OUTPUT_DIR / f"fold_{fold_number:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # 6.2. Apprentissage du classifieur Khiops / SNB
    # --------------------------------------------------------

    classifier = KhiopsClassifier(
        n_trees=0,
        n_pairs=0,
        output_dir=str(fold_dir / "classifier"),
    )

    classifier.fit(X_train, y_train)

    y_pred = classifier.predict(X_test)
    y_proba = classifier.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    metric_row = {
        "fold": fold_number,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": accuracy,
    }

    print(f"\nAccuracy test fold {fold_number} : {accuracy:.4f}")

    # --------------------------------------------------------
    # 6.3. Calcul de l'AUC
    # --------------------------------------------------------
    #
    # Cas binaire :
    #   AUC classique basée sur la probabilité de la seconde classe.
    #
    # Cas multiclasses :
    #   AUC One-vs-Rest, moyenne macro.
    #   Chaque classe est comparée à toutes les autres.
    # --------------------------------------------------------

    classes = classifier.classes_

    if y.nunique() == 2:
        auc = roc_auc_score(
            y_test,
            y_proba[:, 1],
        )
    
        metric_row["auc"] = auc
    
        # Même valeur en binaire : AUC OvR macro = AUC binaire.
        metric_row["auc_ovr_macro"] = auc
        metric_row["auc_ovr_weighted"] = auc
    
        metric_row["auc_type"] = "binary"

    else:

        auc_ovr_macro = roc_auc_score(
            y_test,
            y_proba,
            labels=classes,
            multi_class="ovr",
            average="macro",
        )

        auc_ovr_weighted = roc_auc_score(
            y_test,
            y_proba,
            labels=classes,
            multi_class="ovr",
            average="weighted",
        )

        metric_row["auc_ovr_macro"] = auc_ovr_macro
        metric_row["auc_ovr_weighted"] = auc_ovr_weighted
        metric_row["auc_type"] = "multiclass_ovr"

        print(
            f"AUC multiclasses OVR macro fold {fold_number} : "
            f"{auc_ovr_macro:.4f}"
        )

        print(
            f"AUC multiclasses OVR pondérée fold {fold_number} : "
            f"{auc_ovr_weighted:.4f}"
        )

    all_metrics.append(metric_row)

    # --------------------------------------------------------
    # 6.4. Extraction des poids SNB du fold
    # --------------------------------------------------------

    snb_predictor = classifier.model_report_.modeling_report.get_snb_predictor()
    snb_predictor.init_details()

    weights_df = pd.DataFrame(
        [
            {
                "fold": fold_number,
                "variable_name": variable.name,
                "prepared_name": variable.prepared_name,
                "level": variable.level,
                "weight": variable.weight,
                "importance": variable.importance,
            }
            for variable in snb_predictor.selected_variables
        ]
    ).sort_values("weight", ascending=False)

    print("\nPoids SNB du fold :")
    print(weights_df.to_string(index=False))

    variable_weights = (
        weights_df
        .drop_duplicates(subset="variable_name")
        .set_index("variable_name")["weight"]
        .to_dict()
    )

    weights_path = fold_dir / f"variable_weights_fold_{fold_number:02d}.csv"

    weights_df.to_csv(
        weights_path,
        index=False,
        sep=";",
    )

    all_weights.append(weights_df)

    # --------------------------------------------------------
    # 6.5. Apprentissage du recodeur Conditional Info
    # --------------------------------------------------------
    #
    # Le recodeur est ajusté exclusivement sur le train du fold.
    # Le test est ensuite recodé avec ce même recodeur.
    # --------------------------------------------------------

    encoder = KhiopsEncoder(
        n_trees=0,
        n_pairs=0,
        informative_features_only=True,
        keep_initial_variables=False,
        transform_type_categorical="conditional_info",
        transform_type_numerical="conditional_info",
        transform_type_pairs="conditional_info",
        output_dir=str(fold_dir / "conditional_info_recoder"),
    )

    X_train_ci_array = encoder.fit_transform(X_train, y_train)
    X_test_ci_array = encoder.transform(X_test)

    ci_columns = list(encoder.feature_names_out_)

    X_train_ci = pd.DataFrame(
        X_train_ci_array,
        columns=ci_columns,
        index=X_train.index,
    )

    X_test_ci = pd.DataFrame(
        X_test_ci_array,
        columns=ci_columns,
        index=X_test.index,
    )

    print(f"\nNombre de variables après recodage : {len(ci_columns)}")

    # ============================================================
    # CONTROLE DU SIGNE DES CONDITIONAL INFO KHIOPS
    # ============================================================
    
    print("\nContrôle des Conditional Info brutes :")
    
    print(
        "Minimum train :",
        X_train_ci.to_numpy(dtype=float).min()
    )
    
    print(
        "Maximum train :",
        X_train_ci.to_numpy(dtype=float).max()
    )
    
    display(X_train_ci.head())

    # --------------------------------------------------------
    # 6.6. Correspondance colonne recodée / variable source
    # --------------------------------------------------------

    column_mapping = pd.DataFrame(
        {
            "fold": fold_number,
            "encoded_column": ci_columns,
            "source_variable": [
                infer_source_variable(column, X_train.columns)
                for column in ci_columns
            ],
        }
    )

    column_mapping["weight"] = (
        column_mapping["source_variable"]
        .map(variable_weights)
        .fillna(0.0)
    )

    unmatched = column_mapping[
        column_mapping["source_variable"].isna()
    ]

    if not unmatched.empty:
        print("\nATTENTION : colonnes non associées automatiquement :")
        print(unmatched[["encoded_column"]].to_string(index=False))
        print("Ces variables recevront un poids nul.")

    mapping_path = (
        fold_dir
        / f"conditional_info_column_mapping_fold_{fold_number:02d}.csv"
    )

    column_mapping.to_csv(
        mapping_path,
        index=False,
        sep=";",
    )

    all_mappings.append(column_mapping)

    # --------------------------------------------------------
    # 6.7. Pondération des Conditional Info par les poids SNB
    # --------------------------------------------------------

    weights_vector = (
        column_mapping
        .set_index("encoded_column")
        .loc[ci_columns, "weight"]
        .to_numpy(dtype=float)
    )

    X_train_ci_weighted = X_train_ci.mul(weights_vector, axis=1)
    X_test_ci_weighted = X_test_ci.mul(weights_vector, axis=1)

    X_train_ci_weighted.columns = [
        f"weighted_{column}"
        for column in X_train_ci_weighted.columns
    ]

    X_test_ci_weighted.columns = [
        f"weighted_{column}"
        for column in X_test_ci_weighted.columns
    ]

    # --------------------------------------------------------
    # 6.7 bis. Calcul de log p_w(D) pour l'équation (27)
    # --------------------------------------------------------
    #
    # Le SNB pondéré utilise :
    #
    # p_w(D)
    # =
    # somme_j p(C_j) * produit_m p(D_m | C_j) ** w_m
    #
    # donc :
    #
    # log p_w(D)
    # =
    # logsumexp_j [
    #     log p(C_j)
    #     + somme_m w_m log p(D_m | C_j)
    # ]
    #
    # Les données X_train_ci_weighted et X_test_ci_weighted
    # contiennent déjà les termes :
    #
    # w_m * log p(D_m | C_j)
    #
    # --------------------------------------------------------

    # Classes utilisées par le classifieur Khiops du fold.
    model_classes = list(classifier.classes_)

    n_classes = len(model_classes)

    # Estimation des p(C_j) sur le train du fold.
    #
    # Ces mêmes a priori seront utilisés pour le train ET le test.
    class_priors = (
        y_train
        .value_counts(normalize=True)
        .reindex(model_classes)
    )

    if class_priors.isna().any():
        raise ValueError(
            "Au moins une classe du classifieur est absente "
            "des données train du fold."
        )

    # Toutes les colonnes pondérées disponibles.
    weighted_ci_columns = X_train_ci_weighted.columns.tolist()

    # Association :
    #
    # weighted_Info1P... -> première classe du modèle
    # weighted_Info2P... -> deuxième classe du modèle
    # etc.
    weighted_columns_by_class_index = (
        infer_weighted_ci_columns_by_class_index(
            weighted_ci_columns=weighted_ci_columns,
            n_classes=n_classes,
        )
    )

    # Calcul de log p_w(D) pour les individus train.
    train_global_log_p_D_weighted = (
        compute_weighted_global_log_probability(
            X_ci_weighted=X_train_ci_weighted,
            class_priors=class_priors,
            model_classes=model_classes,
            columns_by_class_index=weighted_columns_by_class_index,
        )
    )

    # Calcul de log p_w(D) pour les individus test.
    test_global_log_p_D_weighted = (
        compute_weighted_global_log_probability(
            X_ci_weighted=X_test_ci_weighted,
            class_priors=class_priors,
            model_classes=model_classes,
            columns_by_class_index=weighted_columns_by_class_index,
        )
    )

    # Dans l'équation (27), le terme est :
    #
    # J * |log p_w(D) - log p_w(D')|
    #
    # Une distance Manhattan sur la coordonnée J * log p_w(D)
    # restitue exactement ce terme.
    train_global_log_p_D_weighted_x_J = (
        n_classes * train_global_log_p_D_weighted
    ).rename("global_log_p_D_weighted_x_J")

    test_global_log_p_D_weighted_x_J = (
        n_classes * test_global_log_p_D_weighted
    ).rename("global_log_p_D_weighted_x_J")

    print("\nTerme global pondéré de l'équation (27) :")
    print(f" - Nombre de classes J : {n_classes}")
    print(
        " - Colonne de distance : "
        "global_log_p_D_weighted_x_J"
    )

    print("\nProbabilités a priori du fold :")

    for class_name, class_prior in class_priors.items():
        print(f" - {class_name} : {class_prior:.6f}")

    # Export des probabilités a priori pour traçabilité.
    priors_df = pd.DataFrame(
        {
            "fold": fold_number,
            "class": class_priors.index,
            "class_prior": class_priors.values,
        }
    )

    priors_path = (
        fold_dir
        / f"{DATASET_NAME}_class_priors_fold_{fold_number:02d}.csv"
    )

    priors_df.to_csv(
        priors_path,
        index=False,
        sep=";",
    )

    all_class_priors.append(priors_df)


    # --------------------------------------------------------
    # 6.8. Reconstruction des datasets avec numéro de fold
    # --------------------------------------------------------
    #
    # Les exports contiennent :
    #
    # 1. Les coordonnées de la distance pondérée NB :
    #    weighted_Info...
    #
    # 2. log p_w(D), uniquement pour contrôle ;
    #
    # 3. J * log p_w(D), à intégrer dans l'équation (27).
    # --------------------------------------------------------

    train_recoded = pd.concat(
        [
            id_train.rename("id").reset_index(drop=True),

            pd.Series(
                fold_number,
                index=range(len(id_train)),
                name="fold",
            ),

            y_train.rename(TARGET_COL).reset_index(drop=True),

            # Coordonnées :
            # w_m * log p(D_m | C_j)
            X_train_ci_weighted.reset_index(drop=True),

            # Valeur de contrôle :
            # log p_w(D)
            train_global_log_p_D_weighted.reset_index(drop=True),

            # Coordonnée utilisée dans l'équation (27) :
            # J * log p_w(D)
            train_global_log_p_D_weighted_x_J.reset_index(drop=True),
        ],
        axis=1,
    )

    test_recoded = pd.concat(
        [
            id_test.rename("id").reset_index(drop=True),

            pd.Series(
                fold_number,
                index=range(len(id_test)),
                name="fold",
            ),

            y_test.rename(TARGET_COL).reset_index(drop=True),

            # Coordonnées :
            # w_m * log p(D_m | C_j)
            X_test_ci_weighted.reset_index(drop=True),

            # Valeur de contrôle :
            # log p_w(D)
            test_global_log_p_D_weighted.reset_index(drop=True),

            # Coordonnée utilisée dans l'équation (27) :
            # J * log p_w(D)
            test_global_log_p_D_weighted_x_J.reset_index(drop=True),
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # 6.9. Exports du fold
    # --------------------------------------------------------

    train_output_path = (
        fold_dir
        / f"{DATASET_NAME}_train_fold_{fold_number:02d}_conditional_info_weighted.csv"
    )

    test_output_path = (
        fold_dir
        / f"{DATASET_NAME}_test_fold_{fold_number:02d}_conditional_info_weighted.csv"
    )

    train_recoded.to_csv(
        train_output_path,
        index=False,
        sep=";",
    )

    test_recoded.to_csv(
        test_output_path,
        index=False,
        sep=";",
    )

    all_train_recoded.append(train_recoded)
    all_test_recoded.append(test_recoded)
                            
    # --------------------------------------------------------
    # Export de contrôle des log-probabilités globales pondérées
    # --------------------------------------------------------

    train_global_probability_df = pd.concat(
        [
            id_train.rename("id").reset_index(drop=True),

            pd.Series(
                fold_number,
                index=range(len(id_train)),
                name="fold",
            ),

            y_train.rename(TARGET_COL).reset_index(drop=True),

            train_global_log_p_D_weighted.reset_index(drop=True),

            train_global_log_p_D_weighted_x_J.reset_index(drop=True),

            pd.Series(
                "train",
                index=range(len(id_train)),
                name="split",
            ),
        ],
        axis=1,
    )

    test_global_probability_df = pd.concat(
        [
            id_test.rename("id").reset_index(drop=True),

            pd.Series(
                fold_number,
                index=range(len(id_test)),
                name="fold",
            ),

            y_test.rename(TARGET_COL).reset_index(drop=True),

            test_global_log_p_D_weighted.reset_index(drop=True),

            test_global_log_p_D_weighted_x_J.reset_index(drop=True),

            pd.Series(
                "test",
                index=range(len(id_test)),
                name="split",
            ),
        ],
        axis=1,
    )

    global_probability_fold_df = pd.concat(
        [
            train_global_probability_df,
            test_global_probability_df,
        ],
        ignore_index=True,
    )

    global_probability_path = (
        fold_dir
        / (
            f"{DATASET_NAME}_weighted_global_log_probability_"
            f"fold_{fold_number:02d}.csv"
        )
    )

    global_probability_fold_df.to_csv(
        global_probability_path,
        index=False,
        sep=";",
    )

    all_global_probabilities.append(
        global_probability_fold_df
    )

    
    print("\nExports créés :")
    print(f" - {train_output_path}")
    print(f" - {test_output_path}")
    print(f" - {weights_path}")
    print(f" - {mapping_path}")
    print(f" - {priors_path}")
    print(f" - {global_probability_path}")


In [ ]:
# ============================================================
# 7. EXPORTS GLOBAUX TOUS FOLDS
# ============================================================

weights_all_folds = pd.concat(all_weights, ignore_index=True)
mappings_all_folds = pd.concat(all_mappings, ignore_index=True)
metrics_all_folds = pd.DataFrame(all_metrics)

train_all_folds = pd.concat(
    all_train_recoded,
    ignore_index=True,
    sort=False,
)

test_all_folds = pd.concat(
    all_test_recoded,
    ignore_index=True,
    sort=False,
)

global_probabilities_all_folds = pd.concat(
    all_global_probabilities,
    ignore_index=True,
)

class_priors_all_folds = pd.concat(
    all_class_priors,
    ignore_index=True,
)

weights_all_path = OUTPUT_DIR / f"{DATASET_NAME}_variable_weights_all_folds.csv"

mappings_all_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_conditional_info_column_mapping_all_folds.csv"
)

metrics_path = OUTPUT_DIR / f"{DATASET_NAME}_metrics_10_folds.csv"

train_all_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_train_all_folds_conditional_info_weighted.csv"
)

test_all_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_test_all_folds_conditional_info_weighted.csv"
)

global_probabilities_all_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_weighted_global_log_probability_all_folds.csv"
)

class_priors_all_path = (
    OUTPUT_DIR
    / f"{DATASET_NAME}_class_priors_all_folds.csv"
)

weights_all_folds.to_csv(
    weights_all_path,
    index=False,
    sep=";",
)

mappings_all_folds.to_csv(
    mappings_all_path,
    index=False,
    sep=";",
)

metrics_all_folds.to_csv(
    metrics_path,
    index=False,
    sep=";",
)

train_all_folds.to_csv(
    train_all_path,
    index=False,
    sep=";",
)

test_all_folds.to_csv(
    test_all_path,
    index=False,
    sep=";",
)

global_probabilities_all_folds.to_csv(
    global_probabilities_all_path,
    index=False,
    sep=";",
)

class_priors_all_folds.to_csv(
    class_priors_all_path,
    index=False,
    sep=";",
)

In [ ]:
# ============================================================
# 8. SYNTHESE FINALE
# ============================================================

print("\n" + "=" * 75)
print("VALIDATION CROISEE A 10 FOLDS TERMINEE")
print("=" * 75)

print("\nMétriques par fold :")
print(metrics_all_folds.to_string(index=False))

print("\nMoyennes des métriques numériques :")
print(metrics_all_folds.mean(numeric_only=True).to_string())

print("\nExports globaux créés :")
print(f" - {weights_all_path}")
print(f" - {mappings_all_path}")
print(f" - {metrics_path}")
print(f" - {train_all_path}")
print(f" - {test_all_path}")
print(f" - {global_probabilities_all_path}")
print(f" - {class_priors_all_path}")

print("\nAperçu du jeu de test global :")
print(test_all_folds.head())

